# CSCN8020 Assignment 1 - Reinforcement Learning Programming

**Student name:** Emmanuel Ihejiamaizu  
**Student ID:** 908005  
**Course:** CSCN8020 - Reinforcement Learning Programming  
**Repository:** `https://github.com/chooksemmanuel/CSCN8020_Assignment1.git`

This notebook contains the full solution for all four assignment problems. It combines Markdown theory, manual calculations, object-oriented Python implementation, algorithm outputs, logging, and comparison tables.

## Assignment assumptions and reading notes

The assignment asks for four reinforcement learning tasks: an MDP design for a pick-and-place robot, two value-iteration updates for a 2x2 Gridworld, value iteration and in-place value iteration for a 5x5 Gridworld, and off-policy Monte Carlo with importance sampling for the same 5x5 environment.

I use the following assumptions consistently:

1. **Discount factor:** \(\gamma = 0.9\), as suggested for the Monte Carlo problem and used consistently across the gridworld tasks.
2. **Reward convention:** the reward is based on the state reached after taking an action. For example, moving into the goal gives \(+10\), moving into a grey state gives \(-5\), and moving into a regular state gives \(-1\).
3. **Invalid action convention:** if an action hits a wall, the agent remains in the same state and receives the reward of that same state.
4. **Terminal convention:** once the agent reaches the goal in the 5x5 Gridworld, the episode ends. The goal is displayed with value \(+10\), but future rewards after entering the terminal state are not double-counted.
5. **Action correction:** the 5x5 problem statement lists `right, down, down, up`. I interpret the repeated `down` as a typo and use the standard four actions: `right`, `down`, `left`, `up`.

## Imports, logging, and reusable objects

The code is organized using object-oriented design. The environment, policies, agents, and utilities are separated into files under the `src/` folder.

Main classes used:

- `DeterministicGridWorldEnv`: environment/model dynamics.
- `ValueIterationAgent`: synchronous value iteration.
- `InPlaceValueIterationAgent`: in-place value iteration.
- `OffPolicyMonteCarloAgent`: off-policy Monte Carlo control with weighted importance sampling.
- `RandomPolicy`: behavior policy for Monte Carlo exploration.
- `configure_logger`: creates the required execution log file.

In [9]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.getcwd())

from src.environments import create_2x2_gridworld, create_5x5_gridworld
from src.agents import ValueIterationAgent, InPlaceValueIterationAgent, OffPolicyMonteCarloAgent
from src.utils import configure_logger, values_to_dataframe, policy_to_dataframe

logger = configure_logger("logs/sample_execution.log")
np.set_printoptions(precision=3, suppress=True)

# Problem 1 - Pick-and-Place Robot MDP Design

A pick-and-place robot repeatedly moves an object from a pickup location to a target location. Since the assignment says the robot must learn movements that are **fast and smooth**, the agent should control low-level motor commands and observe the robot's physical state.

## MDP definition

An MDP can be written as:

\[
\mathcal{M} = (S, A, P, R, \gamma)
\]

where:

- \(S\) is the state space.
- \(A\) is the action space.
- \(P(s'|s,a)\) is the transition model.
- \(R(s,a,s')\) is the reward function.
- \(\gamma\) is the discount factor.

## State space

A useful state should include the information needed to decide the next robot movement:

\[
s_t = [q_t, \dot{q}_t, g_t, p_{object}, p_{target}, d_{gripper,object}, d_{object,target}, c_t]
\]

Where:

- \(q_t\): joint positions of the robot arm.
- \(\dot{q}_t\): joint velocities.
- \(g_t\): gripper state, such as open, closing, or holding object.
- \(p_{object}\): object position.
- \(p_{target}\): target/bin position.
- \(d_{gripper,object}\): distance from gripper to object.
- \(d_{object,target}\): distance from object to target.
- \(c_t\): collision or safety/proximity information.

This state representation is appropriate because the robot needs both position and velocity information to move smoothly, not just to reach the object.

## Action space

Because the agent controls the motors directly, the action can be continuous motor torque or velocity commands:

\[
a_t = [\tau_1, \tau_2, ..., \tau_n, u_{gripper}]
\]

Where:

- \(\tau_i\): motor torque or velocity command for joint \(i\).
- \(u_{gripper}\): gripper open/close command.

This is a continuous-control reinforcement learning problem, not a simple discrete grid movement problem.

## Transition model

The transition model describes how the robot moves after applying a motor command:

\[
P(s_{t+1}|s_t,a_t)
\]

In a real robot, this transition depends on physics: inertia, joint limits, friction, object contact, gripper success, and possible collisions. The transition may be simulated during training before testing on the real robot.

## Reward design

The reward should encourage success, speed, and smoothness:

\[
r_t = 100I_{placed} + 20I_{grasped} - d_{gripper,object} - d_{object,target} - 0.01\|a_t\|^2 - 0.05\|\Delta a_t\|^2 - 50I_{collision} - 1
\]

Meaning:

- `+100` if the object is successfully placed.
- `+20` if the object is grasped.
- Negative distance penalties guide the robot toward the object and target.
- Action penalty discourages unnecessary force.
- Smoothness penalty discourages jerky movements.
- Collision penalty protects the robot and workspace.
- Time-step penalty encourages faster completion.

## Talking points

**A. Key RL feature:** The key feature is MDP design. The robot learns by observing state, taking motor actions, receiving reward, and improving its policy over repeated trials.

**B. Implementation/testing challenge:** The difficult part is reward design. If success reward is too large and smoothness penalty is too small, the robot may learn fast but jerky movements. If smoothness penalty is too large, it may become too slow.

**C. Why this is reinforcement learning:** This is RL because an agent interacts with an environment over time. The robot does not learn from labeled examples. It learns a policy that maximizes expected return through state, action, reward, and future consequence.

# Problem 2 - 2x2 Gridworld Manual Value Iteration

The grid is:

| State | Reward |
|---|---:|
| \(s_1\) | 5 |
| \(s_2\) | 10 |
| \(s_3\) | 1 |
| \(s_4\) | 2 |

Grid layout:

|  |  |
|---|---|
| \(s_1\) | \(s_2\) |
| \(s_3\) | \(s_4\) |

Actions are `up`, `down`, `left`, and `right`. Invalid actions keep the agent in the same state.

I use:

\[
V_0(s) = 0 \quad \text{for all states}
\]

and:

\[
\gamma = 0.9
\]

The Bellman optimality update is:

\[
V_{k+1}(s) = \max_a \left[R(s') + \gamma V_k(s')\right]
\]

where \(s'\) is the next state reached after taking action \(a\).

## Iteration 1

Initial values:

\[
V_0(s_1)=0, \quad V_0(s_2)=0, \quad V_0(s_3)=0, \quad V_0(s_4)=0
\]

### Update for \(s_1\)

From \(s_1\):

- `up` hits wall: stay in \(s_1\), reward = 5
- `down` goes to \(s_3\), reward = 1
- `left` hits wall: stay in \(s_1\), reward = 5
- `right` goes to \(s_2\), reward = 10

\[
V_1(s_1)=\max(5+0.9(0), 1+0.9(0), 5+0.9(0), 10+0.9(0)) = 10
\]

### Update for \(s_2\)

From \(s_2\):

- `up` hits wall: stay in \(s_2\), reward = 10
- `down` goes to \(s_4\), reward = 2
- `left` goes to \(s_1\), reward = 5
- `right` hits wall: stay in \(s_2\), reward = 10

\[
V_1(s_2)=\max(10,2,5,10)=10
\]

### Update for \(s_3\)

From \(s_3\):

- `up` goes to \(s_1\), reward = 5
- `down` hits wall: stay in \(s_3\), reward = 1
- `left` hits wall: stay in \(s_3\), reward = 1
- `right` goes to \(s_4\), reward = 2

\[
V_1(s_3)=\max(5,1,1,2)=5
\]

### Update for \(s_4\)

From \(s_4\):

- `up` goes to \(s_2\), reward = 10
- `down` hits wall: stay in \(s_4\), reward = 2
- `left` goes to \(s_3\), reward = 1
- `right` hits wall: stay in \(s_4\), reward = 2

\[
V_1(s_4)=\max(10,2,1,2)=10
\]

So after Iteration 1:

\[
V_1 =
\begin{bmatrix}
10 & 10 \\
5 & 10
\end{bmatrix}
\]

Greedy policy after Iteration 1:

\[
\pi_1(s_1)=right, \quad \pi_1(s_2)=up/right, \quad \pi_1(s_3)=up, \quad \pi_1(s_4)=up
\]

## Iteration 2

Now use \(V_1\) to compute \(V_2\).

### Update for \(s_1\)

\[
V_2(s_1)=\max(5+0.9(10), 1+0.9(5), 5+0.9(10), 10+0.9(10))
\]

\[
V_2(s_1)=\max(14,5.5,14,19)=19
\]

### Update for \(s_2\)

\[
V_2(s_2)=\max(10+0.9(10), 2+0.9(10), 5+0.9(10), 10+0.9(10))
\]

\[
V_2(s_2)=\max(19,11,14,19)=19
\]

### Update for \(s_3\)

\[
V_2(s_3)=\max(5+0.9(10), 1+0.9(5), 1+0.9(5), 2+0.9(10))
\]

\[
V_2(s_3)=\max(14,5.5,5.5,11)=14
\]

### Update for \(s_4\)

\[
V_2(s_4)=\max(10+0.9(10), 2+0.9(10), 1+0.9(5), 2+0.9(10))
\]

\[
V_2(s_4)=\max(19,11,5.5,11)=19
\]

So after Iteration 2:

\[
V_2 =
\begin{bmatrix}
19 & 19 \\
14 & 19
\end{bmatrix}
\]

## Problem 2 code verification

The original problem asks for the manual process without code. However, the updated assignment requirements also ask for a code implementation or verification in the notebook. The next cell verifies the manual calculations.

In [10]:
grid2 = create_2x2_gridworld()
vi_2_iteration = ValueIterationAgent(
    env=grid2,
    gamma=0.9,
    theta=0.0,
    max_iterations=2,
    logger=logger
).run()

print("Value function after two iterations:")
display(values_to_dataframe(vi_2_iteration.values, 2, 2))

print("Greedy policy after two iterations:")
display(policy_to_dataframe(grid2.render_policy(vi_2_iteration.policy)))

2026-06-10 20:31:14,346 | INFO | Starting synchronous value iteration: gamma=0.9 theta=0.0
2026-06-10 20:31:14,348 | INFO | Value Iteration iteration=1 delta=10.00000000
2026-06-10 20:31:14,350 | INFO | Value Iteration iteration=2 delta=9.00000000
2026-06-10 20:31:14,352 | INFO | Finished synchronous value iteration in 2 iterations and 0.006331 seconds
2026-06-10 20:31:14,354 | INFO | Final values: [19.0, 19.0, 14.0, 19.0]


Value function after two iterations:


,col 0,col 1
row 0,19.0,19.0
row 1,14.0,19.0


Greedy policy after two iterations:


,col 0,col 1
row 0,→,→
row 1,↑,↑


## Problem 2 talking points

**A. Key RL feature:** The key feature is the Bellman optimality update. Each state value is updated by checking all possible actions and choosing the action with the highest expected return.

**B. Implementation/testing challenge:** The main challenge is the reward convention. I explicitly used the reward of the next state so the manual calculation and code match.

**C. Why this is reinforcement learning:** This is RL because the values depend on future interaction between an agent and environment. The agent improves its policy by evaluating state-action consequences, not by learning from labeled input-output examples.

# Problem 3 - 5x5 Gridworld Value Iteration and In-Place Variation

The 5x5 Gridworld contains:

- Terminal/goal state: \(s_{4,4}\)
- Grey states: \(s_{2,2}\), \(s_{3,0}\), and \(s_{0,4}\)
- Goal reward: \(+10\)
- Grey-state reward: \(-5\)
- Regular-state reward: \(-1\)

The dynamic programming method uses the full environment model. This means it knows what next state and reward will result from each action.

In [11]:
grid5 = create_5x5_gridworld()

print("Reward grid used for Problem 3 and Problem 4:")
display(values_to_dataframe(grid5.reward_grid().reshape(-1), 5, 5))

Reward grid used for Problem 3 and Problem 4:


,col 0,col 1,col 2,col 3,col 4
row 0,-1.0,-1.0,-1.0,-1.0,-5.0
row 1,-1.0,-1.0,-1.0,-1.0,-1.0
row 2,-1.0,-1.0,-5.0,-1.0,-1.0
row 3,-5.0,-1.0,-1.0,-1.0,-1.0
row 4,-1.0,-1.0,-1.0,-1.0,10.0


## Standard value iteration

Synchronous value iteration uses two arrays:

- `values`: values from the previous sweep.
- `new_values`: values being computed for the current sweep.

This prevents a state from using newly updated values from the same sweep.

In [12]:
standard_vi = ValueIterationAgent(
    env=grid5,
    gamma=0.9,
    theta=1e-6,
    max_iterations=1000,
    logger=logger
).run()

print(f"Synchronous value iteration converged in {standard_vi.iterations} iterations.")
print(f"Runtime: {standard_vi.elapsed_seconds:.6f} seconds")

print("Optimal state-value function V*:")
display(values_to_dataframe(standard_vi.values, 5, 5))

print("Optimal greedy policy π*:")
display(policy_to_dataframe(grid5.render_policy(standard_vi.policy)))

2026-06-10 20:31:14,397 | INFO | Starting synchronous value iteration: gamma=0.9 theta=1e-06
2026-06-10 20:31:14,399 | INFO | Value Iteration iteration=1 delta=10.00000000
2026-06-10 20:31:14,401 | INFO | Value Iteration iteration=2 delta=9.00000000
2026-06-10 20:31:14,403 | INFO | Value Iteration iteration=3 delta=8.10000000
2026-06-10 20:31:14,405 | INFO | Value Iteration iteration=4 delta=7.29000000
2026-06-10 20:31:14,408 | INFO | Value Iteration iteration=5 delta=6.56100000
2026-06-10 20:31:14,412 | INFO | Finished synchronous value iteration in 9 iterations and 0.013017 seconds
2026-06-10 20:31:14,413 | INFO | Final values: [-0.434, 0.629, 1.81, 3.122, 4.58, 0.629, 1.81, 3.122, 4.58, 6.2, 1.81, 3.122, 4.58, 6.2, 8.0, 3.122, 4.58, 6.2, 8.0, 10.0, 4.58, 6.2, 8.0, 10.0, 10.0]


Synchronous value iteration converged in 9 iterations.
Runtime: 0.013017 seconds
Optimal state-value function V*:


,col 0,col 1,col 2,col 3,col 4
row 0,-0.434,0.629,1.810,3.122,4.58
row 1,0.629,1.810,3.122,4.580,6.20
row 2,1.810,3.122,4.580,6.200,8.00
row 3,3.122,4.580,6.200,8.000,10.00
row 4,4.580,6.200,8.000,10.000,10.00


Optimal greedy policy π*:


,col 0,col 1,col 2,col 3,col 4
row 0,→,→,→,↓,↓
row 1,→,→,→,→,↓
row 2,→,↓,→,→,↓
row 3,→,→,→,→,↓
row 4,→,→,→,→,G


## In-place value iteration

In-place value iteration uses one value array. As soon as a state's value is updated, the new value is available to later states in the same sweep.

This can sometimes converge faster, but the result should still match the same optimal value function and policy for this deterministic gridworld.

In [13]:
in_place_vi = InPlaceValueIterationAgent(
    env=grid5,
    gamma=0.9,
    theta=1e-6,
    max_iterations=1000,
    logger=logger
).run()

print(f"In-place value iteration converged in {in_place_vi.iterations} iterations.")
print(f"Runtime: {in_place_vi.elapsed_seconds:.6f} seconds")

print("In-place state-value function:")
display(values_to_dataframe(in_place_vi.values, 5, 5))

print("In-place greedy policy:")
display(policy_to_dataframe(grid5.render_policy(in_place_vi.policy)))

2026-06-10 20:31:14,497 | INFO | Starting in-place value iteration: gamma=0.9 theta=1e-06
2026-06-10 20:31:14,501 | INFO | In-place Value Iteration iteration=1 delta=10.00000000
2026-06-10 20:31:14,503 | INFO | In-place Value Iteration iteration=2 delta=9.00000000
2026-06-10 20:31:14,506 | INFO | In-place Value Iteration iteration=3 delta=8.10000000
2026-06-10 20:31:14,515 | INFO | In-place Value Iteration iteration=4 delta=7.29000000
2026-06-10 20:31:14,518 | INFO | In-place Value Iteration iteration=5 delta=6.56100000
2026-06-10 20:31:14,524 | INFO | Finished in-place value iteration in 9 iterations and 0.025769 seconds
2026-06-10 20:31:14,525 | INFO | Final in-place values: [-0.434, 0.629, 1.81, 3.122, 4.58, 0.629, 1.81, 3.122, 4.58, 6.2, 1.81, 3.122, 4.58, 6.2, 8.0, 3.122, 4.58, 6.2, 8.0, 10.0, 4.58, 6.2, 8.0, 10.0, 10.0]


In-place value iteration converged in 9 iterations.
Runtime: 0.025769 seconds
In-place state-value function:


,col 0,col 1,col 2,col 3,col 4
row 0,-0.434,0.629,1.810,3.122,4.58
row 1,0.629,1.810,3.122,4.580,6.20
row 2,1.810,3.122,4.580,6.200,8.00
row 3,3.122,4.580,6.200,8.000,10.00
row 4,4.580,6.200,8.000,10.000,10.00


In-place greedy policy:


,col 0,col 1,col 2,col 3,col 4
row 0,→,→,→,↓,↓
row 1,→,→,→,→,↓
row 2,→,↓,→,→,↓
row 3,→,→,→,→,↓
row 4,→,→,→,→,G


In [14]:
max_difference = np.max(np.abs(standard_vi.values - in_place_vi.values))
policy_match = np.array_equal(standard_vi.policy, in_place_vi.policy)

comparison_vi = pd.DataFrame({
    "Method": ["Synchronous Value Iteration", "In-Place Value Iteration"],
    "Iterations/Sweeps": [standard_vi.iterations, in_place_vi.iterations],
    "Runtime seconds": [standard_vi.elapsed_seconds, in_place_vi.elapsed_seconds],
    "Complexity": ["O(iterations × states × actions)", "O(iterations × states × actions)"],
})

display(comparison_vi)
print(f"Maximum absolute value difference: {max_difference:.8f}")
print(f"Do the policies match? {policy_match}")

,Method,Iterations/Sweeps,Runtime seconds,Complexity
0,Synchronous Value Iteration,9,0.013017,O(iterations × states × actions)
1,In-Place Value Iteration,9,0.025769,O(iterations × states × actions)


Maximum absolute value difference: 0.00000000
Do the policies match? True


## Problem 3 comments on performance and complexity

Both standard and in-place value iteration perform Bellman backups over every state and every action. For this gridworld, there are 25 states and 4 actions, so one full sweep checks about \(25 \times 4 = 100\) state-action possibilities.

The computational complexity is:

\[
O(K|S||A|)
\]

where \(K\) is the number of iterations, \(|S|\) is the number of states, and \(|A|\) is the number of actions.

Value iteration does **not** use episodes. It uses dynamic programming sweeps. Therefore, for Problem 3, I report the number of iterations/sweeps instead of number of episodes.

## Problem 3 talking points

**A. Key RL feature:** The key feature is value iteration. It repeatedly applies the Bellman optimality backup to estimate the optimal state-value function \(V^*\).

**B. Implementation/testing challenge:** The main challenge was handling terminal and invalid states consistently. The goal state should end the episode, and invalid actions should keep the agent in the same state.

**C. Why this is reinforcement learning:** This is RL because the algorithm computes a policy based on states, actions, rewards, transitions, and discounted future return. It is not supervised learning because there are no labeled correct moves for each state.

# Problem 4 - Off-Policy Monte Carlo with Importance Sampling

Problem 4 uses the same 5x5 Gridworld, but now the agent estimates values from sampled episodes instead of sweeping through the full model.

The method is **off-policy** because two policies are involved:

- **Behavior policy \(b(a|s)\):** random policy used to generate exploratory episodes.
- **Target policy \(\pi(a|s)\):** greedy policy improved from the learned action-value estimates.

The algorithm follows the weighted importance-sampling idea from Sutton and Barto's off-policy Monte Carlo control pseudocode.

For an episode, returns are processed backward:

\[
G \leftarrow \gamma G + R_{t+1}
\]

The importance-sampling weight corrects the mismatch between the policy used to generate the episode and the policy being evaluated/improved:

\[
W \leftarrow W \times \frac{1}{b(A_t|S_t)}
\]

Since the behavior policy is uniform random over four actions:

\[
b(a|s)=0.25
\]

In [15]:
mc_agent = OffPolicyMonteCarloAgent(
    env=grid5,
    gamma=0.9,
    episodes=5000,
    max_steps_per_episode=100,
    seed=7,
    logger=logger
)
mc_result = mc_agent.run()

print(f"Off-policy Monte Carlo completed {mc_result.episodes} episodes.")
print(f"Runtime: {mc_result.elapsed_seconds:.6f} seconds")
print(f"Visited states with weighted updates: {mc_result.visited_state_count}")

print("Monte Carlo estimated state-value function:")
display(values_to_dataframe(mc_result.values, 5, 5))

print("Monte Carlo greedy target policy:")
display(policy_to_dataframe(grid5.render_policy(mc_result.policy)))

2026-06-10 20:31:14,604 | INFO | Starting off-policy Monte Carlo: episodes=5000 gamma=0.9
2026-06-10 20:31:14,608 | INFO | MC episode=1 length=100
2026-06-10 20:31:14,618 | INFO | MC episode=10 length=23
2026-06-10 20:31:14,709 | INFO | MC episode=100 length=100
2026-06-10 20:31:15,502 | INFO | MC episode=1000 length=55
2026-06-10 20:31:18,174 | INFO | MC episode=5000 length=1
2026-06-10 20:31:18,175 | INFO | Finished off-policy MC in 3.571052 seconds
2026-06-10 20:31:18,177 | INFO | MC estimated values: [-0.438, 0.62, 1.764, 3.023, 4.495, 0.484, 1.723, 3.046, 4.533, 6.13, 1.708, 3.04, 4.492, 6.129, 7.964, 3.085, 4.533, 6.153, 7.971, 10.0, 4.532, 5.895, 7.962, 10.0, 10.0]


Off-policy Monte Carlo completed 5000 episodes.
Runtime: 3.571052 seconds
Visited states with weighted updates: 24
Monte Carlo estimated state-value function:


,col 0,col 1,col 2,col 3,col 4
row 0,-0.438,0.620,1.764,3.023,4.495
row 1,0.484,1.723,3.046,4.533,6.130
row 2,1.708,3.040,4.492,6.129,7.964
row 3,3.085,4.533,6.153,7.971,10.000
row 4,4.532,5.895,7.962,10.000,10.000


Monte Carlo greedy target policy:


,col 0,col 1,col 2,col 3,col 4
row 0,→,↓,↓,↓,↓
row 1,→,→,→,↓,↓
row 2,→,↓,→,→,↓
row 3,↓,→,→,↓,↓
row 4,→,→,→,→,G


In [16]:
absolute_errors = np.abs(standard_vi.values - mc_result.values)
comparison_mc = pd.DataFrame({
    "Method": ["Value Iteration", "Off-Policy Monte Carlo"],
    "Uses full model?": ["Yes", "No"],
    "Main unit of work": ["Sweeps/iterations", "Episodes"],
    "Count": [standard_vi.iterations, mc_result.episodes],
    "Runtime seconds": [standard_vi.elapsed_seconds, mc_result.elapsed_seconds],
    "Complexity": ["O(K × |S| × |A|)", "O(episodes × episode length)"],
})

display(comparison_mc)
print(f"Mean absolute difference from VI values: {np.mean(absolute_errors):.4f}")
print(f"Maximum absolute difference from VI values: {np.max(absolute_errors):.4f}")

,Method,Uses full model?,Main unit of work,Count,Runtime seconds,Complexity
0,Value Iteration,Yes,Sweeps/iterations,9,0.013017,O(K × |S| × |A|)
1,Off-Policy Monte Carlo,No,Episodes,5000,3.571052,O(episodes × episode length)


Mean absolute difference from VI values: 0.0639
Maximum absolute difference from VI values: 0.3047


## Problem 4 comparison comments

Value iteration is faster and more exact in this small gridworld because it has access to the environment model. It can directly check every action from every state.

Monte Carlo does not need the full transition model. Instead, it learns from generated episodes. This makes it more flexible for situations where the model is unknown, but it usually needs many episodes and its estimates can be noisy.

The Monte Carlo value estimates are close to the value-iteration solution, but they are not expected to match perfectly after a finite number of random episodes. Increasing the number of episodes generally improves the estimate.

## Problem 4 talking points

**A. Key RL feature:** The key feature is off-policy learning with importance sampling. The behavior policy explores randomly, while the target policy is updated greedily from learned action values.

**B. Implementation/testing challenge:** The main challenge was avoiding infinite episodes. Since a random policy may wander for a long time, I used a maximum episode length.

**C. Why this is reinforcement learning:** This is RL because the agent learns from experience generated by interaction with the environment. It uses rewards, returns, episodes, policies, and value estimates instead of labeled training examples.

# Final summary

This notebook completed all four assignment problems in a single runnable file. It defined a robot-control task as an MDP, manually solved two iterations of value iteration for a 2x2 Gridworld, implemented standard and in-place value iteration for a 5x5 Gridworld, and implemented off-policy Monte Carlo with importance sampling. The code uses object-oriented design, logs execution progress, and compares algorithms by runtime, iterations or episodes, and computational complexity.

# References

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- CSCN8020 course materials for MDPs, Dynamic Programming, and Monte Carlo methods.
- Gymnasium-style environment design: `reset`, `step`, state, action, reward, terminated, and truncated interaction pattern.